# 동아대 대학원 QML 강의 — 3일차 (11/27)
## 실습(1.5시간): QGAN 생성·증강·비교 + MediQ-GAN 개념 데모

**오늘의 핵심**: 2일차에 준비한 전처리 데이터와 회로 구성요소를 바탕으로, 강사가 10월에 사전 학습시켜 둔
체크포인트를 불러와 생성·증강·분류 비교까지 전체 파이프라인을 실행하고 결과를 해석함.
라이브 학습은 진행하지 않음(이유는 2번 섹션 참고).

**실습 자산 배포 방식**: 사전학습 체크포인트와 2일차 전처리 데이터는 강의 GitHub 저장소
(https://github.com/hyunhp/skin_disease_qml)의 `day3/assets/` 폴더에 들어 있음. 1번 섹션에서 저장소를 `git clone`으로
내려받아 사용함 — 로그인·권한 설정이 필요 없고, 자산 총 용량은 수 MB 수준임(원본 6GB는 불필요).

**시간 배분(1.5시간)**: 복습+조립(15분, Colab 사용법은 0번 섹션 참고) → 체크포인트 로드+생성(20분) → 증강+CNN비교(20분)
→ MediQ-GAN 개념 데모(20분) → 고전대비비교 토론(15분)

이후 부산 지사 견학 + 회사 소개로 이어짐.


## 0. Google Colab 사용법 (빠른 복습)

2일차와 동일하게 **Google Colab**에서 진행함. 2일차에 참석하지 못했거나 Colab이 처음이면 2일차 노트북 **"0. Google Colab 시작하기"**(계정 만들기 포함)를 먼저 확인함.

### 0-1. 시작 전 확인
- Google 계정으로 https://colab.research.google.com 에 로그인되어 있는지 확인함 (학교·회사 계정이 막혀 있으면 개인 Gmail 사용).
- 노트북을 연 뒤 **`파일(File) → Drive에 사본 저장(Save a copy in Drive)`** 실행 — 사본을 만들어야 실행 결과가 저장됨.
- "이 노트북은 Google에서 작성하지 않았습니다(This notebook was not authored by Google)" 경고가 뜨면 `무시하고 계속(Run anyway)` 클릭.
- **런타임은 기본 CPU로 충분함** — GPU로 바꿀 필요 없음(양자회로 시뮬레이션과 8×8 CNN 모두 CPU로 실행됨).

### 0-2. 자주 쓰는 조작
| 하고 싶은 것 | 방법 |
|---|---|
| 셀 실행 | **Shift + Enter**, 또는 셀 왼쪽 ▶ 버튼 |
| 실행 중단 | 셀 왼쪽 ■ 버튼, 또는 `런타임(Runtime) → 실행 중단(Interrupt execution)` |
| 처음부터 전부 실행 | `런타임(Runtime) → 모두 실행(Run all)` |
| 꼬였을 때 초기화 | `런타임(Runtime) → 세션 다시 시작(Restart session)` 후 1번 섹션부터 다시 실행 |

### 0-3. 오늘 실습에서 알아둘 점
- **2일차 런타임은 이미 종료된 상태임** — 2일차에 설치한 라이브러리·변수는 남아 있지 않으므로 오늘도 1번 섹션 설치 셀부터 **위에서 아래로 순서대로** 실행함(건너뛰면 `NameError` 발생).
- **로그인이 필요 없음** — 실습 자산은 공개 GitHub 저장소에서 `git clone`으로 받으며, 수 MB라 수 초면 끝남(원본 6GB는 받지 않음).
- 오늘은 **학습 없이** 사전학습 체크포인트를 불러와 실행만 하므로 대부분의 셀이 수 초 안에 끝남. 4번 섹션 MediQ-GAN 데모(16큐빗×5개 회로)만 수 초~십여 초 걸릴 수 있음(정상).
- 수업 도중 연결이 끊기면(유휴 상태가 길 때 발생) 다시 연결한 뒤 1번 섹션부터 다시 실행하면 됨 — 전체를 다시 실행해도 1~2분 이내임.
- 오류가 나면 **오류 메시지 마지막 줄**을 먼저 확인함 — 대부분 앞 셀 미실행 또는 연결 끊김이 원인임.

## 1. 2일차 복습 + 실습 자산 다운로드 + 서브제너레이터 패치 조립 (15분)

2일차에 만든 서브제너레이터 회로 1개를 여러 개(patch 방식)로 확장함. 논문(Table 3)은 16개를
사용했으나, Colab 무료 시뮬레이터로 실시간 처리 가능한 규모로 축소해 사용함.

서브제너레이터 각각은 이미지의 일부(patch)만 담당하며, patch들을 이어붙여 전체 이미지를 구성함.
따라서 `n_generators × 2^n_data_qubits = 전체 픽셀수` 관계가 성립해야 함(체크포인트의 `config`에
저장된 값을 그대로 사용하므로 직접 계산할 필요는 없음).

**2일차 회로와 같은 측정 방식**: 보조큐빗까지 측정해 **보조큐빗이 0인 경우의 확률만** patch로 사용함
(논문의 사후선택 post-selection에 해당). 이렇게 하면 patch마다 밝기 총합이 0~1 사이에서 학습되므로,
배경(검정)이 많은 위·아래 patch와 병변이 있는 가운데 patch를 다르게 표현할 수 있음.

**2일차 회로와 달라진 점(큐빗 수)**: 2일차에는 데이터 큐빗 6개 + 보조 1개 = **7큐빗 회로 1개**로 8×8(64픽셀) 전체를 만들었음.
오늘 체크포인트는 데이터 큐빗 4개 + 보조 1개 = **5큐빗 회로 4개**가 각각 16픽셀(8×8 이미지의 2줄)씩 나눠 만듦(4 × 2⁴ = 64).
회로 하나를 작게 만들고 개수를 늘리는 것이 patch 방식의 핵심임 — 회로 구조(RY 인코딩 → RY+CZ 반복 → 보조큐빗=0 선택)는 2일차와 동일함.

In [ ]:
!pip install -q pennylane pennylane-lightning koreanize-matplotlib
import os, math, json, time
import numpy as np, torch, torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
# 그래프 한글 표시 — Colab 기본 폰트에는 한글이 없어 설정하지 않으면 그래프 글자가 네모(□)로 깨짐
try:
    import koreanize_matplotlib  # 나눔고딕 폰트 등록 + 기본 폰트 지정
except Exception as e:
    print("한글 폰트 설정 실패 — 그래프의 한글만 깨지고 실습 진행에는 지장 없음:", e)

# 강의 저장소 — 실습 자산(2일차 전처리 데이터 + 사전학습 체크포인트)이 day3/assets/에 들어 있음
REPO_URL = "https://github.com/hyunhp/skin_disease_qml.git"
REPO_DIR = "/content/skin_disease_qml"

In [ ]:
# 저장소 내려받기 (이미 받아둔 경우에는 최신 상태로 갱신만 함)
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q --depth 1 {REPO_URL} {REPO_DIR}
ASSET_DIR = f"{REPO_DIR}/day3/assets"
if not os.path.exists(f"{ASSET_DIR}/qgan_df_8px.pt"):
    raise RuntimeError("실습 자산이 없음 — 위 git clone 출력의 오류 메시지(네트워크 등)를 확인하고 이 셀을 다시 실행할 것")
print("실습 자산 경로:", ASSET_DIR)
print("포함 항목:", sorted(os.listdir(ASSET_DIR)))

df_imgs = np.load(f"{ASSET_DIR}/df_preprocessed.npy")
nv_imgs = np.load(f"{ASSET_DIR}/nv_preprocessed.npy")
IMG_SIZE = df_imgs.shape[1]
print(f"2일차 전처리 데이터 로드: DF {df_imgs.shape}, NV {nv_imgs.shape}, IMG_SIZE={IMG_SIZE}")

# (선택) 2일차에 본인이 Drive에 저장한 데이터로 진행하려면 아래 주석을 해제함
# from google.colab import drive; drive.mount('/content/drive')
# df_imgs = np.load("/content/drive/MyDrive/동아대_QML강의/df_preprocessed.npy")
# nv_imgs = np.load("/content/drive/MyDrive/동아대_QML강의/nv_preprocessed.npy")

In [ ]:
class PatchQuantumGenerator(nn.Module):
    """서브제너레이터 앙상블 — 2일차의 단일 회로를 N개로 확장해 patch 방식으로 이미지 조립.

    인자 기본값은 체크포인트의 config로 항상 덮어쓰므로 참고용임."""
    def __init__(self, n_generators=4, n_data_qubits=4, n_ancillas=1, q_depth=3):
        super().__init__()
        self.n_generators = n_generators
        self.n_data_qubits = n_data_qubits
        self.n_qubits = n_data_qubits + n_ancillas
        self.q_depth = q_depth
        self.patch_dim = 2 ** n_data_qubits
        self.anc_stride = 2 ** n_ancillas  # 보조큐빗이 마지막 wire → '보조큐빗=0' 항목은 이 간격마다 위치함

        dev = qml.device("default.qubit", wires=self.n_qubits)  # backprop 미분에는 lightning.qubit이 아니라 default.qubit
        n_qubits = self.n_qubits

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(noise, weights):
            for i in range(n_qubits):
                qml.RY(noise[i], wires=i)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])
            # 보조큐빗까지 전체 측정 → forward에서 '보조큐빗=0' 항목만 patch로 사용 (논문의 사후선택에 해당)
            return qml.probs(wires=range(n_qubits))

        self.circuit = circuit
        self.q_params = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, self.n_qubits) * math.pi) for _ in range(n_generators)
        ])

    def forward(self, batch_size, device="cpu"):
        all_patches = []
        for gen_idx in range(self.n_generators):
            weights = self.q_params[gen_idx]
            # [::anc_stride] = 보조큐빗이 0인 경우의 확률만 선택 → patch 밝기 총합(0~1)도 학습 대상이 됨
            batch_patches = [self.circuit(torch.rand(self.n_qubits, device=device) * math.pi, weights)[::self.anc_stride]
                              for _ in range(batch_size)]
            all_patches.append(torch.stack(batch_patches))
        return torch.cat(all_patches, dim=1).float()

    @property
    def output_dim(self):
        return self.n_generators * self.patch_dim

print("PatchQuantumGenerator 클래스 정의 완료 — 아직 랜덤 초기화 상태(학습 전)임")

## 2. 사전학습 체크포인트 로드 + 생성 (20분)

**지금 학습을 진행하지 않는 이유**: 논문 실측 학습시간은 **1~1.5시간(2000 epoch)**으로, 오늘 실습
전체 시간(1.5시간)과 맞먹음. 따라서 강사가 10월 준비 기간에 학습을 완료해뒀으며, 지금은
그 결과물(체크포인트)을 불러와 **추론(생성)만** 실행함.

체크포인트(`qgan_df_8px.pt`)는 1번 셀에서 내려받은 `ASSET_DIR` 안에 포함되어 있음.

In [ ]:
ckpt_path = f"{ASSET_DIR}/qgan_df_{IMG_SIZE}px.pt"
checkpoint = torch.load(ckpt_path, map_location="cpu")
cfg = checkpoint["config"]
print("불러온 체크포인트 설정:", cfg)

gen = PatchQuantumGenerator(n_generators=cfg["n_generators"], n_data_qubits=cfg["n_data_qubits"],
                             n_ancillas=cfg["n_ancillas"], q_depth=cfg["q_depth"])
gen.load_state_dict(checkpoint["generator_state_dict"])
gen.eval()
print("체크포인트 로드 완료 — 학습이 완료된 생성자임 (2일차의 랜덤 출력과 비교해볼 것)")

In [ ]:
# 이미지 생성 (추론만 수행, 학습 없음)
with torch.no_grad():
    generated = gen(batch_size=8).numpy().reshape(8, IMG_SIZE, IMG_SIZE)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img in zip(axes.flat, generated):
    ax.imshow(img, cmap="gray"); ax.axis("off")
plt.suptitle("QGAN이 생성한 DF(피부섬유종) 합성 이미지 — 사전학습 체크포인트 기반")
plt.tight_layout(); plt.show()

# 실제 DF 이미지와 나란히 비교
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax, img in zip(axes, df_imgs[:4]):
    ax.imshow(img, cmap="gray"); ax.axis("off")
plt.suptitle("(비교용) 실제 DF 원본 이미지")
plt.tight_layout(); plt.show()

## 3. 데이터 증강 + CNN 분류기 비교 (20분)

생성된 이미지로 소수클래스(DF)를 오버샘플링한 뒤, **사전학습된 CNN 분류기 2종**(증강 전/증강 후)의
성능을 비교함. 이 단계도 학습이 아니라 **불러와서 평가만** 수행함 — CNN 학습도 10월에 사전 완료함.

**정직한 비교를 위한 조건**: 두 모델은 **동일한 held-out 검증셋**으로 평가되었으며, 검증셋에는
합성(QGAN 생성) 이미지를 포함하지 않음. 증강은 학습셋에만 적용함 — 검증셋에 합성 이미지가 섞이거나
학습에 쓴 데이터로 평가하면 증강 효과가 실제보다 부풀려져 보이게 됨.

In [ ]:
class PaperCNNClassifier(nn.Module):
    def __init__(self, in_channels=1, img_size=64):
        super().__init__()
        chs_full = [in_channels, 16, 32, 64, 128]
        n_layers, size = 0, img_size
        while size >= 2 and n_layers < 4:
            size //= 2; n_layers += 1
        n_layers = max(n_layers, 1)
        chs = chs_full[: n_layers + 1]
        layers, size = [], img_size
        for i in range(n_layers):
            do_pool = size >= 2
            layers += [nn.Conv2d(chs[i], chs[i+1], 3, 1, 1), nn.BatchNorm2d(chs[i+1]), nn.ReLU()]
            if do_pool:
                layers.append(nn.MaxPool2d(2, 2)); size //= 2
        self.conv = nn.Sequential(*layers)
        flat_dim = chs[-1] * size * size
        self.fc = nn.Sequential(nn.Linear(flat_dim, 128), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(128, 32), nn.ReLU(), nn.Dropout(0.3),
                                 nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))

def load_cnn(path, img_size):
    ckpt = torch.load(path, map_location="cpu")
    model = PaperCNNClassifier(in_channels=1, img_size=img_size)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model

cnn_before = load_cnn(f"{ASSET_DIR}/cnn_before.pt", IMG_SIZE)
cnn_after = load_cnn(f"{ASSET_DIR}/cnn_after.pt", IMG_SIZE)
print("CNN 분류기 2종(증강 전/후) 로드 완료")

In [ ]:
with open(f"{ASSET_DIR}/comparison_result.json", encoding="utf-8") as f:
    result = json.load(f)

print(f"소수클래스: {result['minority_class']} (실제 {result['n_minority_real']}장 + 합성 {result['n_synthetic']}장)")
if "n_minority_val" in result:
    print(f"검증셋(held-out): 소수 {result['n_minority_val']}장 + 다수 {result['n_majority_val']}장 — 합성 이미지 미포함")
print()
print(f"[증강 전] val accuracy={result['before']['accuracy']:.3f}  val AUC-ROC={result['before']['auc_roc']:.3f}")
print(f"[증강 후] val accuracy={result['after']['accuracy']:.3f}  val AUC-ROC={result['after']['auc_roc']:.3f}")
print()
print("논문 실측치(Table 6, DF vs NV): 증강 전 AUC-ROC=0.953 → 증강 후 0.970 (참고 비교용)")
print("우리 실습은 데이터 수·epoch·서브제너레이터 수를 축소했으므로 논문 수치와 다를 수 있음 — 5번 섹션 토론 주제임")

plt.figure(figsize=(5, 4))
plt.bar(["증강 전", "증강 후"], [result['before']['auc_roc'], result['after']['auc_roc']], color=["#B8C8DE", "#0066CC"])
plt.ylabel("AUC-ROC"); plt.ylim(0.0, 1.0); plt.title("증강 전 vs 증강 후 — held-out 검증셋 기준")
plt.show()

## 4. MediQ-GAN 개념 데모 (20분)

**MediQ-GAN**(arXiv:2506.21015)은 지금까지 본 것과 다른 종류의 하이브리드임.
- 지금까지(HAM10000 QGAN 논문): **생성자 전체가 양자** + 판별자만 고전
- MediQ-GAN: **생성자 내부에서** 고전 레이어와 양자 레이어를 **skip-connection으로 융합**

양자 스트림은 원 논문·공식 레포와 동일한 구성을 그대로 사용함(1일차 슬라이드 17과 같은 흐름):
- 양자 쪽 입력을 4×4=16칸(site)으로 보고, 칸 [0, 4, 8, 11, 15] 5곳만 서브제너레이터 5개에 하나씩 입력함
- 회로(16큐빗, 8층): **RY→RX 각도 인코딩 → [학습 RY + 인접 CZ] 반복 → PauliX 기댓값 측정**
- 큐빗 16개의 기댓값을 평균해 칸마다 스칼라 1개 → 4×4 맵에 배치(나머지 11칸은 0) → 1×1 conv로 16채널 투영 후 고전 스트림과 concat

1회 forward에 수 초 소요되나 Colab 라이브 시연에는 문제없음(학습이 아니라 구조 확인용 1회 실행).
다만 classical encoder/decoder는 원 논문의 conv 대신 단순 MLP로 단순화했고, 입력도 실제 64×64 RGB가
아니라 2일차에서 만든 8×8(64차원) 저해상도 벡터임.

In [ ]:
class ToyDualStreamGenerator(nn.Module):
    """MediQ-GAN Fig.2 구조: 입력 특징을 절반씩 나눠 고전/양자 스트림으로 병렬 처리한 뒤
    concat(skip-connection에 해당)으로 융합. 양자 스트림은 공식 레포(mediq-gan.py의
    quantum_circuit·_quantum_branch_8to4)와 같은 방식:
      - 양자 쪽 절반을 4×4=16 site로 보고, site별 벡터를 q_align(Linear)으로 n_qubits차원 토큰으로 맞춤
      - linspace(0,15,n_generators).round()로 고른 site([0,4,8,11,15])를 서브제너레이터마다 하나씩 입력
      - 회로: RY(x)→RX(x) 인코딩 → [학습 RY + 인접 CZ] × q_depth → PauliX 기댓값
      - 큐빗 방향 평균 → site당 스칼라 1개 → 4×4 맵의 해당 칸에 배치(나머지 11칸은 0)
      - 1×1 conv(칸별 독립, 학습됨)로 n_generators채널 → q_ch채널 투영 후 고전 스트림과 concat
    (q_depth: 논문 그림 기준 8, 레포 CLI 기본값은 6)"""

    def __init__(self, in_dim=64, n_qubits=16, q_depth=8, n_generators=5, out_dim=64,
                 q_ch=16, dev_name="lightning.qubit"):
        super().__init__()
        assert in_dim % 2 == 0, "고전/양자 스트림으로 절반씩 나누려면 in_dim이 짝수여야 함"
        self.half = in_dim // 2
        assert self.half % 16 == 0, "양자 쪽 절반을 4×4=16 site로 나누려면 in_dim/2가 16의 배수여야 함"
        self.site_dim = self.half // 16
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        self.n_generators = n_generators
        self.site_idx = torch.linspace(0, 15, steps=n_generators).round().long().tolist()

        # 고전 스트림 — MediQ-GAN의 "classical encoder" 역할(단순화: conv 대신 MLP)
        self.classical_stream = nn.Sequential(
            nn.Linear(self.half, 32), nn.ReLU(),
            nn.Linear(32, self.half),
        )

        # 양자 스트림 — site별 벡터를 큐빗 수에 맞추는 aligner(레포의 q_align)
        self.q_align = nn.Linear(self.site_dim, n_qubits)

        dev = qml.device(dev_name, wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="adjoint")
        def q_circuit(x, weights):
            # 공식 레포와 동일: 같은 값으로 RY→RX 인코딩, 학습 RY + 인접 CZ 반복, X-basis 측정
            for i in range(n_qubits):
                qml.RY(x[i], wires=i)
                qml.RX(x[i], wires=i)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]

        self.q_circuit = q_circuit
        self.q_weights = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, n_qubits) * math.pi) for _ in range(n_generators)
        ])
        # 레포의 proj_qexp: 5개 스칼라 맵(4×4) → q_ch채널, 1×1 conv라 칸별 독립 처리
        self.proj_qexp = nn.Conv2d(n_generators, q_ch, 1)

        # 융합 후 디코더 — skip-connection으로 합쳐진 특징을 최종 출력으로 (MediQ-GAN의 decoder conv에 해당)
        fused_dim = self.half + q_ch * 16
        self.decoder = nn.Sequential(
            nn.Linear(fused_dim, 32), nn.ReLU(),
            nn.Linear(32, out_dim), nn.Tanh(),
        )

    def quantum_stream(self, z_quantum):
        B = z_quantum.shape[0]
        sites = z_quantum.view(B, 16, self.site_dim)       # 4×4 = 16 site
        tokens = self.q_align(sites)                        # (B, 16, n_qubits)
        q_maps = torch.zeros(B, self.n_generators, 4, 4, dtype=z_quantum.dtype)
        for g, si in enumerate(self.site_idx):              # 서브제너레이터 g ↔ site si
            outs = torch.stack([torch.stack(self.q_circuit(tokens[b, si], self.q_weights[g]))
                                for b in range(B)])         # (B, n_qubits)
            y, x = divmod(si, 4)
            q_maps[:, g, y, x] = outs.mean(dim=1).to(z_quantum.dtype)  # 큐빗 평균 → 스칼라
        return self.proj_qexp(q_maps).flatten(1)            # (B, q_ch*16)

    def forward(self, z):
        # z: (batch, in_dim) — 잠재벡터를 절반씩 스트림에 분배 (MediQ-GAN Fig.2: "feature map split")
        z_classical, z_quantum = z[:, :self.half], z[:, self.half:]
        classical_feat = self.classical_stream(z_classical)  # (batch, half)
        quantum_feat = self.quantum_stream(z_quantum)        # (batch, q_ch*16)
        fused = torch.cat([classical_feat, quantum_feat], dim=1)  # skip-connection 융합
        return self.decoder(fused)

toy_gen = ToyDualStreamGenerator(in_dim=IMG_SIZE*IMG_SIZE, n_qubits=16, q_depth=8, n_generators=5, out_dim=IMG_SIZE*IMG_SIZE)
z = torch.rand(1, IMG_SIZE*IMG_SIZE)
t0 = time.time()
demo_out = toy_gen(z)
print(f"dual-stream 융합 출력 shape: {demo_out.shape} (forward {time.time()-t0:.1f}초) — 고전 스트림 + 양자 스트림(16큐빗×5개 → site별 스칼라 → 1×1 conv)이 하나로 융합됨")
print("(학습 전 랜덤 가중치이므로 의미있는 이미지는 아님 — 구조 확인용. 양자회로는 공식 레포와 동일: RY→RX 인코딩, 학습 RY+CZ, PauliX 측정)")

### MediQ-GAN 원 논문 실측 결과 (참고표 — 우리가 재현한 것이 아니라 논문 수치 그대로 인용)

| 데이터셋 | 내용 | 결과 |
|---|---|---|
| ISIC 2019 | 피부경검사 8클래스 | 소수클래스(df, vasc, scc)에서 모든 고전 베이스라인 대비 최고/차상위 FID |
| ODIR-5k | 안저사진 | 두 백본 모두에서 최고 성능 |
| RetinaMNIST | 다운샘플 안저사진 | 5개 클래스 중 4개에서 최저(최고품질) FID |

비교 대상: DCGAN, WGAN-GP, StyleGAN2-ADA, FastGAN, **diffusion 모델(DiT)**, 그리고 이전 세대 하이브리드
모델 MosaiQ까지 포함. IBM 실기기(`ibm_marrakesh`)에서 추론 검증도 완료함.

**시사점**: "생성자 전체를 양자로" 만드는 것보다, "고전이 잘하는 부분(전역 구조)은 고전에 맡기고
양자는 디테일 보강에만 사용하는" 레이어 단위 하이브리드가 더 높은 해상도·더 좋은 품질로 확장 가능함을
보여줌 — 오늘 실습한 HAM10000 QGAN(생성자 전체 양자)의 자연스러운 발전 방향임.


## 5. 고전 대비 비교 + 향후 방안 토론 (15분)

HAM10000 QGAN 논문 Table 11 — DF vs NV 증강 기법 비교 (논문 수치 그대로 인용):

| 증강 기법 | Raw 정확도 | Raw AUC | Segment 정확도 | Segment AUC |
|---|---|---|---|---|
| 기하학적 변형 | 0.87 | 0.82 | 0.92 | 0.96 |
| 믹싱/erasing | 0.84 | 0.86 | 0.88 | 0.94 |
| **고전 GAN** | **0.98** | **0.98** | 0.95 | 0.97 |
| **QGAN** | 0.89 | 0.87 | **0.96** | 0.97(동률) |

### 토론 질문
1. **raw 이미지에서는 고전 GAN이 우세하고, segmentation 이미지에서는 QGAN이 우세한 이유는?**
   (힌트: 큐빗 수 제약 → 저차원 표현을 다뤄야 함 → segmentation이 "정보를 압축"해주는 역할)
2. 오늘 재현한 결과와 논문 실측치가 다르다면, 원인으로 무엇을 생각할 수 있는가?
   (서브제너레이터 개수 축소, 학습 epoch 수, 데이터 샘플 수 등)
3. MediQ-GAN처럼 "일부만 양자"인 구조가 고해상도로 더 잘 확장되는 이유는?
4. **2번 섹션에서 생성된 이미지가 실제 DF 이미지(가운데 밝은 병변)와 달리 흩어진 점 패턴에 가까운 이유는?**
   (힌트: 서브제너레이터 1개 = 5큐빗·depth 4 회로가 8×8 이미지의 2줄(16픽셀)을 담당함 / 매번 무작위 노이즈 각도로
   시작함 / 학습 epoch 300은 논문 2,000의 15% 수준임 / 판별자가 너무 빨리 이기면 생성자가 배울 신호가 사라짐)
   - 참고(강사 사전 실험): 보조큐빗 없이 데이터 큐빗만 측정하는 방식은 patch마다 밝기 총합이 1로 고정되어
     "배경만 있는 patch"를 표현하지 못했고, 판별자가 이 차이만으로 항상 이겨 학습이 무너졌음. 보조큐빗=0인 경우만
     쓰는 방식(현재 코드)으로 바꾸자 생성 이미지의 평균이 실제와 닮아졌으나(상관계수 0.00→0.49), 개별 이미지는
     여전히 병변 모양이 아님 — **회로 구조(측정 방식) 하나가 학습 성패를 좌우**한 사례임.
   - 증강 전/후 AUC 차이가 작거나 뒤바뀌는 것도 같은 맥락임 — 검증셋 소수클래스가 20여 장이라 ±0.02 정도는
     우연으로도 흔들림. "개선됐다"고 말하려면 무엇이 더 필요한가?

### 향후 방안
- 논문 원래 스펙(16 서브제너레이터, 7+1큐빗)은 큐빗 수로 보면 지금도 시뮬레이터로 충분히 다룰 수 있는 규모임 — 실제 제약은 학습 시간(논문 기준 2,000 epoch에 1~1.5시간)과 학습 안정성(판별자 우세 문제)임. 실기(QPU)에서 학습까지 하려면 parameter-shift 실행 비용(1일차 슬라이드 26)이 추가 제약이 됨
- 레이어 단위 하이브리드(MediQ-GAN 방향)가 실용화에 더 가까운 경로로 판단됨
- "양자가 항상 우세하다"가 아니라 "특정 전처리·구조 조건에서 고전과 동등하거나 근소 우위" —
  이것이 현재 공개 문헌이 정직하게 뒷받침하는 결론임

---
## 3일차 마무리 체크리스트
- [ ] 저장소(git clone)에서 실습 자산 다운로드 완료
- [ ] 사전학습 QGAN 체크포인트 로드 + 합성 이미지 생성 확인
- [ ] 증강 전/후 CNN 비교 결과 확인 (held-out 검증셋 기준임을 이해)
- [ ] MediQ-GAN dual-stream 구조 forward 실행 확인
- [ ] 고전 대비 비교 토론 참여
